In [53]:
import pandas as pd
import numpy as np
import os
import re
import ast
import json

from pathlib import Path
from collections import Counter

In [54]:
# Notebook location:
# ml_training/resume_extraction/raw_data_processing/

BASE_DIR = Path("..")
RAW_DIR = BASE_DIR / "raw data"

# Input files
RESUME_FILE = RAW_DIR / "resumes" / "training_data.csv"
ESCO_FILE = RAW_DIR / "esco" / "skills_en.csv"
ONET_FILE = RAW_DIR / "onet" / "software_skills.csv"
SKILLS_FILE = RAW_DIR / "skills" / "skills_list.csv"

# Output directory
OUTPUT_DIR = Path("final_training_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Resume file :", RESUME_FILE)
print("ESCO file   :", ESCO_FILE)
print("O*NET file  :", ONET_FILE)
print("Skills file :", SKILLS_FILE)
print("Output dir  :", OUTPUT_DIR)

Resume file : ..\raw data\resumes\training_data.csv
ESCO file   : ..\raw data\esco\skills_en.csv
O*NET file  : ..\raw data\onet\software_skills.csv
Skills file : ..\raw data\skills\skills_list.csv
Output dir  : final_training_data


In [55]:
files_to_check = {
    "Resume dataset": RESUME_FILE,
    "ESCO": ESCO_FILE,
    "O*NET": ONET_FILE,
    "Skills list": SKILLS_FILE
}

for name, path in files_to_check.items():
    print(f"{name:20} -> {'FOUND' if path.exists() else 'NOT FOUND'}")

Resume dataset       -> FOUND
ESCO                 -> FOUND
O*NET                -> FOUND
Skills list          -> FOUND


In [56]:
resumes = pd.read_csv(RESUME_FILE)

print("Resume dataset shape:", resumes.shape)
print("\nColumns:")
print(resumes.columns.tolist())

print("\nFirst 5 rows:")
display(resumes.head())

Resume dataset shape: (10000, 7)

Columns:
['Resume ID', 'Resume Text', 'Education', 'Experience Years', 'Skills', 'Job Role', 'Category']

First 5 rows:


,Resume ID,Resume Text,Education,Experience Years,Skills,Job Role,Category
0,R000000,Education: Bachelor's in Computer Science Expe...,Bachelor's in Computer Science,2,Market Research|Maven|Java|REST API|Spring Boo...,Java Backend Developer,Technology
1,R000001,Education: Master's in Microbiology Experience...,Master's in Microbiology,3,Agile|Data Analysis|Precision|Problem Solving|...,Microbiologist,Science & Research
2,R000002,Education: Apprenticeship Experience: 2 years ...,Apprenticeship,2,React|Customer Service|Technical Knowledge|Plu...,Plumber,Skilled Trades
3,R000003,Education: Bachelor's in Computer Science Expe...,Bachelor's in Computer Science,0,MySQL|Query Optimization|Symfony|Terraform|PHP...,PHP Developer,Technology
4,R000004,Education: Bachelor's in Design Experience: 3 ...,Bachelor's in Design,3,Problem Solving|Docker|Creativity|Scrum|Ansibl...,Art Director,Creative & Design


In [57]:
def find_column(df, possible_names):
    normalized = {
        str(col).strip().lower().replace("_", " ").replace("-", " "): col
        for col in df.columns
    }

    for name in possible_names:
        key = name.lower().replace("_", " ").replace("-", " ")
        if key in normalized:
            return normalized[key]

    return None


TEXT_COL = find_column(
    resumes,
    [
        "Resume Text",
        "resume_text",
        "Resume",
        "resume",
        "Text",
        "text"
    ]
)

SKILLS_COL = find_column(
    resumes,
    [
        "Skills",
        "skills",
        "Skill",
        "skill"
    ]
)

ID_COL = find_column(
    resumes,
    [
        "Resume ID",
        "resume_id",
        "ID",
        "id"
    ]
)

print("Resume text column :", TEXT_COL)
print("Skills column      :", SKILLS_COL)
print("ID column          :", ID_COL)

Resume text column : Resume Text
Skills column      : Skills
ID column          : Resume ID


In [58]:
if TEXT_COL is None:
    raise ValueError("Resume text column was not found.")

resumes = resumes.copy()

# Convert resume text to string
resumes[TEXT_COL] = resumes[TEXT_COL].fillna("").astype(str)

# Basic text cleaning
def clean_text(text):
    text = str(text)
    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


resumes["resume_text_clean"] = resumes[TEXT_COL].apply(clean_text)

# Remove completely empty resumes
resumes = resumes[
    resumes["resume_text_clean"].str.len() > 0
].copy()

# Reset index
resumes.reset_index(drop=True, inplace=True)

print("Clean resume dataset shape:", resumes.shape)
print("Empty resumes:", (resumes["resume_text_clean"] == "").sum())

Clean resume dataset shape: (10000, 8)
Empty resumes: 0


In [59]:
esco = pd.read_csv(ESCO_FILE)

print("ESCO shape:", esco.shape)

print("\nESCO columns:")
print(esco.columns.tolist())

display(esco.head())

ESCO shape: (13960, 13)

ESCO columns:
['conceptType', 'conceptUri', 'skillType', 'reuseLevel', 'preferredLabel', 'altLabels', 'hiddenLabels', 'status', 'modifiedDate', 'scopeNote', 'definition', 'inScheme', 'description']


,conceptType,conceptUri,skillType,reuseLevel,preferredLabel,altLabels,hiddenLabels,status,modifiedDate,scopeNote,definition,inScheme,description
0,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0005c151-5b5a...,skill/competence,sector-specific,manage musical staff,manage music staff\ncoordinate duties of music...,NaN,released,2023-11-30T15:53:37.136Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Assign and manage staff tasks in areas such as...
1,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00064735-8fad...,skill/competence,occupation-specific,supervise correctional procedures,manage prison procedures\nmonitor correctional...,NaN,released,2023-11-30T15:04:00.689Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Supervise the operations of a correctional fac...
2,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/000709ed-2be5...,skill/competence,sector-specific,apply anti-oppressive practices,make use of anti-oppressive practices\nuse ant...,NaN,released,2023-11-28T10:45:53.54Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Identify oppression in societies, economies, c..."
3,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/0007bdc2-dd15...,skill/competence,sector-specific,control compliance of railway vehicles regulat...,checking compliance with rolling stock regulat...,NaN,released,2023-11-30T16:29:18.273Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,"Inspect rolling stock, components and systems ..."
4,KnowledgeSkillCompetence,http://data.europa.eu/esco/skill/00090cc1-1f27...,skill/competence,cross-sector,identify available services,determine available services\nclassify availab...,NaN,released,2023-11-28T10:38:49.206Z,NaN,NaN,http://data.europa.eu/esco/concept-scheme/skil...,Identify the different services available for ...


In [60]:
ESCO_LABEL_COL = find_column(
    esco,
    [
        "preferredLabel",
        "preferred_label",
        "Skill Name",
        "skill",
        "label"
    ]
)

ESCO_ALT_COL = find_column(
    esco,
    [
        "altLabels",
        "alt_labels",
        "alternative labels",
        "alternate labels"
    ]
)

ESCO_URI_COL = find_column(
    esco,
    [
        "conceptUri",
        "concept_uri",
        "uri"
    ]
)

print("ESCO main skill column :", ESCO_LABEL_COL)
print("ESCO alternate column  :", ESCO_ALT_COL)
print("ESCO URI column        :", ESCO_URI_COL)

ESCO main skill column : preferredLabel
ESCO alternate column  : altLabels
ESCO URI column        : conceptUri


In [61]:
esco_skills = []

for _, row in esco.iterrows():

    # Main preferred skill name
    if ESCO_LABEL_COL is not None:
        value = row[ESCO_LABEL_COL]

        if pd.notna(value):
            value = str(value).strip()

            if value:
                esco_skills.append(value)

    # Alternative labels
    if ESCO_ALT_COL is not None:
        value = row[ESCO_ALT_COL]

        if pd.notna(value):
            value = str(value)

            # ESCO alternative labels may use comma/newline separators
            parts = re.split(r"[,\n;|]", value)

            for part in parts:
                part = part.strip()

                if part:
                    esco_skills.append(part)


esco_skills = list(set(esco_skills))

print("Unique ESCO skills:", len(esco_skills))
print(esco_skills[:20])

Unique ESCO skills: 99735
['organise materials to tea bag machines', 'follow advanced manufacturing practices', 'teach linguistics', 'shape nails', 'reaction to mining emergencies', 'create subtitles', 'machine translation', 'international duties', 'warm blooded animal organs', 'adjust communication style according to target audience', 'solve information-related issues', 'overseeing of wetlands in project development', 'develop with cloud services', 'photovoltaic panel mounting', 'lace tobacco crops', 'implement airport standards and regulations', 'move filled moulds', 'pharmaceutical industry', 'civil process order', 'orthopaedic goods business']


In [62]:
onet = pd.read_csv(ONET_FILE)

print("O*NET shape:", onet.shape)

print("\nO*NET columns:")
print(onet.columns.tolist())

display(onet.head())

O*NET shape: (31821, 7)

O*NET columns:
['O*NET-SOC Code', 'Title', 'Workplace Example', 'Element ID', 'Element Name', 'Hot Technology', 'In Demand']


,O*NET-SOC Code,Title,Workplace Example,Element ID,Element Name,Hot Technology,In Demand
0,11-1011.00,Chief Executives,Adobe Acrobat,2.E.5.b,Document management software,Y,N
1,11-1011.00,Chief Executives,AdSense Tracker,2.E.6.f,Data base user interface and query software,N,N
2,11-1011.00,Chief Executives,Atlassian JIRA,2.E.5.a,Content workflow software,Y,N
3,11-1011.00,Chief Executives,Blackbaud The Raiser's Edge,2.E.6.c,Customer relationship management CRM software,N,N
4,11-1011.00,Chief Executives,ComputerEase construction accounting software,2.E.2.a,Accounting software,N,N


In [63]:
ONET_SKILL_COL = find_column(
    onet,
    [
        "Element Name",
        "element_name",
        "Skill Name",
        "skill_name",
        "Skill",
        "skill"
    ]
)

print("O*NET skill column:", ONET_SKILL_COL)

if ONET_SKILL_COL is None:
    raise ValueError("Could not find the O*NET skill column.")

O*NET skill column: Element Name


In [64]:
onet_skills = (
    onet[ONET_SKILL_COL]
    .dropna()
    .astype(str)
    .str.strip()
)

onet_skills = [
    skill for skill in onet_skills
    if skill
]

onet_skills = list(set(onet_skills))

print("Unique O*NET skills:", len(onet_skills))
print(onet_skills[:20])

Unique O*NET skills: 134
['Business intelligence and data analysis software', 'Configuration management software', 'Bar coding software', 'Process mapping and design software', 'Communications server software', 'Contact center software', 'Enterprise application integration software', 'Portal server software', 'Internet browser software', 'Operating system software', 'Development environment software', 'Enterprise resource planning ERP software', 'Computer based training software', 'Time accounting software', 'Graphics or photo imaging software', 'Audit software', 'Project management software', 'Video creation and editing software', 'Network security or virtual private network VPN management software', 'Wireless software']


In [65]:
skills_df = pd.read_csv(SKILLS_FILE)

print("Skills list shape:", skills_df.shape)

print("\nColumns:")
print(skills_df.columns.tolist())

display(skills_df.head())

Skills list shape: (120, 2)

Columns:
['Skill Name', 'Category']


,Skill Name,Category
0,Python,Programming
1,Java,Programming
2,JavaScript,Programming
3,C++,Programming
4,C#,Programming


In [66]:
CUSTOM_SKILL_COL = find_column(
    skills_df,
    [
        "Skill Name",
        "skill_name",
        "Skill",
        "skill",
        "Name",
        "name"
    ]
)

print("Custom skill column:", CUSTOM_SKILL_COL)

if CUSTOM_SKILL_COL is None:
    raise ValueError("Could not find the custom skill column.")

Custom skill column: Skill Name


In [67]:
custom_skills = (
    skills_df[CUSTOM_SKILL_COL]
    .dropna()
    .astype(str)
    .str.strip()
)

custom_skills = [
    skill for skill in custom_skills
    if skill
]

custom_skills = list(set(custom_skills))

print("Custom skills:", len(custom_skills))
print(custom_skills[:20])

Custom skills: 115
['Kubernetes', 'MongoDB', 'Business Development', 'Encryption', 'HTML', 'GCP', 'Security Architecture', 'Go', 'Microservices', 'Jupyter', 'Leadership', 'Cassandra', 'Ruby', 'Azure', 'MySQL', 'Adobe Creative Suite', 'Android', 'Firewalls', 'Scrum', 'React Native']


In [68]:
all_skills = (
    esco_skills
    + onet_skills
    + custom_skills
)

print("Total raw skill entries:", len(all_skills))

Total raw skill entries: 99984


In [69]:
def normalize_skill(skill):
    skill = str(skill).lower().strip()

    # Replace different dash types
    skill = skill.replace("–", "-")
    skill = skill.replace("—", "-")

    # Normalize whitespace
    skill = re.sub(r"\s+", " ", skill)

    return skill


skill_vocab = {}

for skill in all_skills:

    original = str(skill).strip()

    if not original:
        continue

    normalized = normalize_skill(original)

    if not normalized:
        continue

    skill_vocab[normalized] = original


print("Unique normalized skills:", len(skill_vocab))

Unique normalized skills: 99891


In [70]:
GENERIC_TERMS = {
    "business",
    "communication",
    "creativity",
    "development",
    "experience",
    "information",
    "knowledge",
    "management",
    "people",
    "problem solving",
    "process",
    "service",
    "services",
    "skills",
    "teamwork",
    "technology",
    "work",
    "working"
}

filtered_skill_vocab = {
    skill: original
    for skill, original in skill_vocab.items()
    if skill not in GENERIC_TERMS
}

print("Before filtering:", len(skill_vocab))
print("After filtering :", len(filtered_skill_vocab))
print("Removed         :", len(skill_vocab) - len(filtered_skill_vocab))

Before filtering: 99891
After filtering : 99887
Removed         : 4


In [71]:
def is_valid_skill(skill):

    skill = skill.strip()

    # Single alphabetic character is usually too ambiguous
    if len(skill) == 1 and skill.isalpha():
        return False

    # Empty
    if not skill:
        return False

    # Must contain at least one alphanumeric character
    if not re.search(r"[a-zA-Z0-9]", skill):
        return False

    return True


filtered_skill_vocab = {
    skill: original
    for skill, original in filtered_skill_vocab.items()
    if is_valid_skill(skill)
}

print("Final vocabulary size:", len(filtered_skill_vocab))

Final vocabulary size: 99886


In [72]:
master_vocabulary = pd.DataFrame({
    "canonical_skill": [
        filtered_skill_vocab[x]
        for x in filtered_skill_vocab
    ],
    "normalized_skill": list(filtered_skill_vocab.keys())
})

master_vocabulary["word_count"] = (
    master_vocabulary["normalized_skill"]
    .str.split()
    .str.len()
)

master_vocabulary = master_vocabulary.sort_values(
    by=["word_count", "normalized_skill"],
    ascending=[False, True]
).reset_index(drop=True)

print("Master vocabulary shape:", master_vocabulary.shape)

display(master_vocabulary.head(20))

Master vocabulary shape: (99886, 3)


,canonical_skill,normalized_skill,word_count
0,create pipeline designs accounting for the coa...,create pipeline designs accounting for the coa...,17
1,ability to comprehend spoken and written Ancie...,ability to comprehend spoken and written ancie...,16
2,ensure navigation equipment is prepared for us...,ensure navigation equipment is prepared for us...,16
3,ability to comprehend spoken and written Limbu...,ability to comprehend spoken and written limbu...,15
4,discuss with social services stakeholders disc...,discuss with social services stakeholders disc...,15
5,ensure safe and efficient flow of air traffic ...,ensure safe and efficient flow of air traffic ...,15
6,ensure tram inspections and operations take pl...,ensure tram inspections and operations take pl...,15
7,help passengers to identify the best train acc...,help passengers to identify the best train acc...,15
8,make assessment of structural integrity of shi...,make assessment of structural integrity of shi...,15
9,perform evaluation of structural integrity of ...,perform evaluation of structural integrity of ...,15


In [73]:
MASTER_CSV = OUTPUT_DIR / "master_skill_vocabulary.csv"
MASTER_PARQUET = OUTPUT_DIR / "master_skill_vocabulary.parquet"

master_vocabulary.to_csv(
    MASTER_CSV,
    index=False
)

master_vocabulary.to_parquet(
    MASTER_PARQUET,
    index=False
)

print("Saved:")
print(MASTER_CSV)
print(MASTER_PARQUET)

Saved:
final_training_data\master_skill_vocabulary.csv
final_training_data\master_skill_vocabulary.parquet


In [74]:
skill_list = list(filtered_skill_vocab.keys())

skill_list = sorted(
    skill_list,
    key=lambda x: (
        len(x.split()),
        len(x)
    ),
    reverse=True
)

print("Skills available for matching:", len(skill_list))
print(skill_list[:30])

Skills available for matching: 99886
['create pipeline designs accounting for the coating solutions related to the goods to be transported through them', 'ensure navigation equipment is prepared for use in navigation operations carry out navigation equipment preparation activities', 'ability to comprehend spoken and written ancient greek and to speak and write in ancient greek', 'discuss with social services stakeholders discuss with social service stakeholders discuss with social service stakeholder', 'ensure tram inspections and operations take place as scheduled through collaboration with tram maintenance department', 'perform evaluation of structural integrity of ship in relation to its functionality for maritime activities', 'make assessment of structural integrity of ship in relation to its functionality for maritime activities', 'ability to comprehend spoken and written limburgish and to speak and write in limburgish limburgian', 'help passengers to identify the best train accor

In [75]:
TOKEN_PATTERN = re.compile(
    r"\w+|[^\w\s]",
    re.UNICODE
)


def tokenize_with_positions(text):
    tokens = []
    
    for match in TOKEN_PATTERN.finditer(text):
        tokens.append({
            "text": match.group(),
            "start": match.start(),
            "end": match.end()
        })
    
    return tokens


def normalize_token(token):
    return token.lower().strip()

In [76]:
# Create a fast lookup dictionary for skills

skill_lookup = {}

for skill in skill_list:
    normalized = normalize_skill(skill)

    # Tokenize the skill
    skill_tokens = tuple(
        re.findall(r"\w+", normalized)
    )

    if skill_tokens:
        skill_lookup[skill_tokens] = skill


# Maximum number of words in any skill
MAX_SKILL_WORDS = max(
    len(x) for x in skill_lookup.keys()
)

print("Skills in lookup:", len(skill_lookup))
print("Maximum skill phrase length:", MAX_SKILL_WORDS)

Skills in lookup: 99650
Maximum skill phrase length: 17


In [77]:
def find_skill_spans_fast(text, skill_lookup, max_skill_words):

    token_info = tokenize_with_positions(text)

    tokens = [
        normalize_token(x["text"])
        for x in token_info
    ]

    matches = []

    n = len(tokens)

    for i in range(n):

        # Try phrases of different lengths
        for length in range(
            1,
            min(max_skill_words, n - i) + 1
        ):

            phrase = tuple(
                tokens[i:i + length]
            )

            if phrase in skill_lookup:

                skill = skill_lookup[phrase]

                matches.append({
                    "skill": skill,
                    "start": token_info[i]["start"],
                    "end": token_info[i + length - 1]["end"]
                })

    # Longest matches first
    matches.sort(
        key=lambda x: (
            x["start"],
            -(x["end"] - x["start"])
        )
    )

    # Remove overlapping matches
    selected = []

    for match in matches:

        overlap = False

        for existing in selected:

            if (
                match["start"] < existing["end"]
                and match["end"] > existing["start"]
            ):
                overlap = True
                break

        if not overlap:
            selected.append(match)

    return selected

In [78]:
def create_bio_labels_fast(
    text,
    skill_lookup,
    max_skill_words
):

    tokens_info = tokenize_with_positions(text)

    tokens = [
        item["text"]
        for item in tokens_info
    ]

    labels = [
        "O"
        for _ in tokens_info
    ]

    matches = find_skill_spans_fast(
        text,
        skill_lookup,
        max_skill_words
    )

    for match in matches:

        matched_indices = []

        for i, token_info in enumerate(tokens_info):

            if (
                token_info["start"] < match["end"]
                and token_info["end"] > match["start"]
            ):
                matched_indices.append(i)

        if matched_indices:

            labels[matched_indices[0]] = "B-SKILL"

            for idx in matched_indices[1:]:
                labels[idx] = "I-SKILL"

    return tokens, labels, matches

In [79]:
import time

start_time = time.time()

test_count = min(100, len(resumes))

for idx in range(test_count):

    text = resumes.loc[
        idx,
        "resume_text_clean"
    ]

    tokens, labels, matches = create_bio_labels_fast(
        text,
        skill_lookup,
        MAX_SKILL_WORDS
    )

elapsed = time.time() - start_time

print(f"Processed {test_count} resumes")
print(f"Time taken: {elapsed:.2f} seconds")

estimated_total = (
    elapsed / test_count
) * len(resumes)

print(
    f"Estimated time for {len(resumes)} resumes: "
    f"{estimated_total:.2f} seconds"
)

print(
    f"Approximately: "
    f"{estimated_total / 60:.2f} minutes"
)

Processed 100 resumes
Time taken: 0.03 seconds
Estimated time for 10000 resumes: 2.80 seconds
Approximately: 0.05 minutes


In [80]:
final_records = []

start_time = time.time()

for idx, row in resumes.iterrows():

    text = row["resume_text_clean"]

    tokens, labels, matches = create_bio_labels_fast(
        text,
        skill_lookup,
        MAX_SKILL_WORDS
    )

    detected_skills = list(
        dict.fromkeys(
            match["skill"]
            for match in matches
        )
    )

    record = {
        "resume_id": (
            row[ID_COL]
            if ID_COL is not None
            else idx
        ),
        "resume_text": text,
        "tokens": tokens,
        "ner_tags": labels,
        "skills": detected_skills,
        "num_skills": len(detected_skills)
    }

    final_records.append(record)

    # Progress every 500 resumes
    if (idx + 1) % 500 == 0:

        elapsed = time.time() - start_time

        print(
            f"Processed {idx + 1}/{len(resumes)} "
            f"| Time: {elapsed:.1f}s"
        )


final_dataset = pd.DataFrame(final_records)

print("\nFinal dataset shape:", final_dataset.shape)

Processed 500/10000 | Time: 0.1s
Processed 1000/10000 | Time: 0.2s
Processed 1500/10000 | Time: 0.3s
Processed 2000/10000 | Time: 0.4s
Processed 2500/10000 | Time: 0.5s
Processed 3000/10000 | Time: 0.6s
Processed 3500/10000 | Time: 0.7s
Processed 4000/10000 | Time: 0.8s
Processed 4500/10000 | Time: 0.9s
Processed 5000/10000 | Time: 1.0s
Processed 5500/10000 | Time: 1.1s
Processed 6000/10000 | Time: 1.2s
Processed 6500/10000 | Time: 1.3s
Processed 7000/10000 | Time: 1.4s
Processed 7500/10000 | Time: 1.5s
Processed 8000/10000 | Time: 1.6s
Processed 8500/10000 | Time: 1.7s
Processed 9000/10000 | Time: 1.8s
Processed 9500/10000 | Time: 1.9s
Processed 10000/10000 | Time: 2.0s

Final dataset shape: (10000, 6)


In [81]:
print(final_dataset.columns.tolist())

display(
    final_dataset[
        [
            "resume_id",
            "skills",
            "num_skills"
        ]
    ].head(10)
)

['resume_id', 'resume_text', 'tokens', 'ner_tags', 'skills', 'num_skills']


,resume_id,skills,num_skills
0,R000000,"[computer science, market research, maven, jav...",8
1,R000001,"[microbiology, agile, data analysis, scala, bu...",5
2,R000002,"[react, customer service, postgresql]",3
3,R000003,"[computer science, mysql, query optimization, ...",8
4,R000004,"[design, docker, scrum, ansible, leadership, d...",6
5,R000005,"[database design, pytorch, deep learning]",3
6,R000006,"[computer science, cassandra, nosql, business ...",5
7,R000007,"[it, sql, etl, javascript, kanban, web accessi...",6
8,R000008,"[swift, physical therapy, php]",3
9,R000009,"[computer science, postgresql, kanban, complia...",5


In [82]:
print("Total resumes:", len(final_dataset))

print(
    "Resumes with at least 1 skill:",
    (final_dataset["num_skills"] > 0).sum()
)

print(
    "Resumes with no detected skills:",
    (final_dataset["num_skills"] == 0).sum()
)

print(
    "Average skills per resume:",
    final_dataset["num_skills"].mean()
)

Total resumes: 10000
Resumes with at least 1 skill: 9998
Resumes with no detected skills: 2
Average skills per resume: 4.8399


In [83]:
sample = final_dataset.iloc[0]

for token, label in zip(
    sample["tokens"],
    sample["ner_tags"]
):
    print(f"{token:25} {label}")

Education                 O
:                         O
Bachelor                  O
'                         O
s                         O
in                        O
Computer                  B-SKILL
Science                   I-SKILL
Experience                O
:                         O
2                         O
years                     O
Skills                    O
:                         O
Market                    B-SKILL
Research                  I-SKILL
,                         O
Maven                     B-SKILL
,                         O
Java                      B-SKILL
,                         O
REST                      B-SKILL
API                       I-SKILL
,                         O
Spring                    O
Boot                      O
,                         O
Microservices             B-SKILL
,                         O
Statistics                B-SKILL
,                         O
SQL                       B-SKILL


In [84]:
label2id = {
    "O": 0,
    "B-SKILL": 1,
    "I-SKILL": 2
}

id2label = {
    0: "O",
    1: "B-SKILL",
    2: "I-SKILL"
}

final_dataset["ner_ids"] = final_dataset["ner_tags"].apply(
    lambda labels: [
        label2id[label]
        for label in labels
    ]
)

print(label2id)

{'O': 0, 'B-SKILL': 1, 'I-SKILL': 2}


In [85]:
training_dataset = final_dataset[
    final_dataset["num_skills"] > 0
].copy()

training_dataset.reset_index(drop=True, inplace=True)

print("Complete dataset:", final_dataset.shape)
print("Training dataset:", training_dataset.shape)

Complete dataset: (10000, 7)
Training dataset: (9998, 7)


In [86]:
FINAL_CSV = OUTPUT_DIR / "final_resume_skill_ner_dataset.csv"

training_dataset.to_csv(
    FINAL_CSV,
    index=False
)

print("Saved:", FINAL_CSV)

Saved: final_training_data\final_resume_skill_ner_dataset.csv


In [1]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

MODEL_PATH = "./skill_ner_model"   # change this to your saved model folder

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

print("Model loaded successfully!")
print("Device:", device)

ValueError: Couldn't instantiate the backend tokenizer from one of: 
(1) a `tokenizers` library serialization file, 
(2) a slow tokenizer instance to convert or 
(3) an equivalent slow tokenizer class to instantiate and convert. 
You need to have sentencepiece or tiktoken installed to convert a slow tokenizer to a fast one.